# Scaling up to CNNs
With the basics out of the way, we can start building larger networks. In this notebook, we will introduce to you how to use convolutional and pooling layers.

In [ ]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron, SpikingGammaOutput
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import compute_linspace_bucket_decay_rates, SpikeConfig
from spikinggamma.dendrite.synapse.conv2d import Conv2dPerNeuron

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Loading the data
We use DVS Gesture for this experiment, with 500 timesteps. Caching is used through tonic's `DiskCachedDataset`. This makes the first epoch much slower.

In [2]:
from load_data import get_dvsgesture_dataset
train_dataset, test_dataset, input_shape, n_outputs, T = get_dvsgesture_dataset(timestep_size=12000)

## Building the network
As you can see, building a CNN is very similar to a fully-connected network. Just swap the Linear layer with a Conv2d layer. It is also possible to insert some pooling layers that are applied to the neuron signal.

In [3]:
# Hyperparameters
n_buckets = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    spike_threshold=0.3
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_buckets, decay_factor)

class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_buckets,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        output_shape_layer1 = (64, 30, 30)
        output_shape_layer2 = (128, 13, 13)
        output_shape_layer3 = (128, 11, 11)
        input_shape_layer4 = (128*5*5,)
        layer4_shape_out = (n_outputs,)

        self.conv1 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer1,
                n_buckets=n_buckets,
                synaptic_function=Conv2dPerNeuron(input_shape[0], 64, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(output_shape_layer1))
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer2,
                n_buckets=n_buckets,
                synaptic_function=Conv2dPerNeuron(64, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(output_shape_layer2))
        )

        self.conv3 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer3,
                n_buckets=n_buckets,
                synaptic_function=Conv2dPerNeuron(128, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(output_shape_layer3))
        )
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer4_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(input_shape_layer4[0], layer4_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction()
        )

        self.out = SpikingGammaOutput(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config)
        )

    def reset(self):
        self.conv1.reset()
        self.conv2.reset()
        self.conv3.reset()
        self.fc.reset()
        self.out.reset()

    def detach(self):
        self.conv1.detach()
        self.conv2.detach()
        self.conv3.detach()
        self.fc.detach()
        self.out.detach()

    def forward(self, s_in):
        y_conv1 = self.conv1(s_in)
        y_pool1 = self.pool1(y_conv1)
        y_conv2 = self.conv2(y_pool1)
        y_conv3 = self.conv3(y_conv2)
        y_pool3 = self.pool3(y_conv3)
        y_pool3 = y_pool3.flatten(start_dim=1)
        y_fc = self.fc(y_pool3)
        y_hat_out, s_out = self.out(y_fc)
        return y_fc, y_hat_out, s_out

## Training the network

In [4]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_buckets, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [5]:
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y, y_hat, s = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.9465: 100%|██████████| 34/34 [13:44<00:00, 24.25s/it]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing

In [6]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        s_list = [] 
        for t in range(data.shape[0]):
            y, y_hat, s = model(data[t])
            s_list.append(s)
        s = torch.stack(s_list, dim=0)
        predictions = s.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 3/3 [02:56<00:00, 58.70s/it]

Test Accuracy: 35.61%
